**BfPortf 01 Simple example**

This example shows how to use [bfportf](https://github.com/answering007/bfportf) to find portfolios with the [Sharpe ratio](https://en.wikipedia.org/wiki/Sharpe_ratio) as the optimized metric. The search strategies are greedy (default), beam and exhaustive.

In [ ]:
# Import section
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option('display.precision', 3)

import bfportf as bf


In [ ]:
# Load data
prices = pd.read_csv(
    'https://raw.githubusercontent.com/answering007/bfportf/master/examples/prices.csv',
    index_col=0,
    parse_dates=['Date'],
)
returns = bf.load_returns(prices)
returns.head()


In [ ]:
# Split data to train and test
df_train, df_test = train_test_split(returns, test_size=0.2, shuffle=False)
print(f'Train shape: {df_train.shape}, Test shape: {df_test.shape}')
print(f'Train start: {str(df_train.index[0])}, end: {str(df_train.index[-1])}')
print(f'Test start: {str(df_test.index[0])}, end: {str(df_test.index[-1])}')


In [ ]:
# Scale returns: every symbol gets the same average absolute return,
# the shares are inverse volatility weights
df_train_scaled, train_weights = bf.scale_return_data(df_train)
df_test_scaled, test_weights = bf.scale_return_data(df_test)


In [ ]:
# Run the search: greedy forward selection with the Sharpe ratio metric
found_portfolios = bf.find_portfolios(
    returns=df_train_scaled,
    metric='sharpe',
    custom_weights=train_weights,
    max_number=3,
    verbose=True,
)
bf.results_to_frame(found_portfolios).head()


In [ ]:
# The best portfolio: found_portfolios[0] is a PortfolioResult object
best = found_portfolios[0]
print(best.symbols)
print(best.weights)

formatting = {
    'PnL': '{:.2%}',
    'CAGR': '{:.2%}',
    'Max drawdown': '{:.2%}',
}

# In-sample results
report = bf.create_report(
    returns=df_train_scaled,
    portfolio_symbols=best.symbols,
    return_weights=train_weights,
    benchmark_symbols=['SPY'],
    custom_formatting=formatting,
)
report


In [ ]:
# Out-of-sample results
report = bf.create_report(
    returns=df_test_scaled,
    portfolio_symbols=best.symbols,
    return_weights=train_weights,
    benchmark_symbols=['SPY'],
    custom_formatting=formatting,
)
report


### Other search strategies

`bf.BeamSearch(width=5)` explores several best partial portfolios at once, `bf.ExhaustiveSearch()` enumerates every combination and returns the true optimum for a given `max_number` (only suitable for small universes, it refuses to run above `max_combinations`).

In [ ]:
small = df_train_scaled[list(df_train_scaled.columns[:10])]
beam = bf.find_portfolios(small, metric='sharpe', max_number=3, strategy=bf.BeamSearch(width=5), verbose=False)
exhaustive = bf.find_portfolios(small, metric='sharpe', max_number=3, strategy=bf.ExhaustiveSearch(), verbose=False)
print('greedy    ', found_portfolios[0].value)
print('beam      ', beam[0].value)
print('exhaustive', exhaustive[0].value)
